# 103 — Postprocess Binary Tiles

Applies per-tile postprocessing to binary prediction tiles:
1. Cultivated land suppression
2. Shadow mask (if available)
3. Morphological cleanup (median, opening, closing)

Skips tiles with no lupine before AND after cleaning.
Saves cleaned tiles to `pp_tiles_t{thresh}/`.

Run after `101_predict.ipynb` and `102_shadow.ipynb`.
Run `104_vectorize.ipynb` after this.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
from config import *
from helpers import *
import time
import gc
import numpy as np

print(f"Config loaded ✓")
print(f"  Mode:       {MODE}")
print(f"  Thresholds: {PIXEL_THRESHOLDS}")
print(f"  Morphology: median={MEDIAN_SIZE}px  opening={OPENING_SIZE}px  closing={CLOSING_SIZE}px")
print(f"  Shadow:     threshold={BRIGHTNESS_THRESH}")
check_required_files()

In [ ]:
# ── Load cultivated land once ─────────────────────────────────────────────────
print("── Loading cultivated land ─────────────────────────────────")
cultivated_gdf = gpd.read_file(CULTIVATED_PATH)
cultivated_gdf["geometry"] = cultivated_gdf.geometry.make_valid()
print(f"  Cultivated polygons: {len(cultivated_gdf)}")

In [ ]:
# Load lupine tile list from 102
tile_list_path = os.path.join(MODEL_OUT_DIR, "lupine_tiles.txt")
if os.path.exists(tile_list_path):
    with open(tile_list_path) as f:
        lupine_tiles = set(f.read().splitlines())
    print(f"  Loaded {len(lupine_tiles)} lupine tiles from list")
else:
    print("  [!] lupine_tiles.txt not found — run 102_shadow.ipynb first")

# Build to_process from lupine tiles
ref_thresh  = PIXEL_THRESHOLDS[0]
pp_existing = set(os.listdir(pp_tile_dir(ref_thresh)))

to_process = [
    fname for fname in lupine_tiles
    if fname.replace(".tif", f"_pp_t{ref_thresh}.tif") not in pp_existing
]

print(f"  Total lupine tiles: {len(lupine_tiles)}")
print(f"  Already done:       {len(lupine_tiles) - len(to_process)}")
print(f"  To process:         {len(to_process)}")

In [ ]:
# ── Postprocess per tile ──────────────────────────────────────────────────────
print("── Postprocessing tiles ────────────────────────────────────")

SHDW_DIR = shadow_tile_dir()
skipped  = 0
removed  = 0
t0       = time.time()

for i, source_fname in enumerate(to_process):
    remaining_count = len(to_process) - i - 1

    for thresh in PIXEL_THRESHOLDS:
        bin_fname = source_fname.replace(".tif", f"_binary_t{thresh}.tif")
        bin_path  = os.path.join(binary_tile_dir(thresh), bin_fname)
        pp_fname  = source_fname.replace(".tif", f"_pp_t{thresh}.tif")
        pp_path   = os.path.join(pp_tile_dir(thresh), pp_fname)
        shd_fname = source_fname.replace(".tif", "_shadow.tif")
        shd_path  = os.path.join(SHDW_DIR, shd_fname)

        if not os.path.exists(bin_path):
            print(f"  [!] Binary tile missing: {bin_fname}")
            continue

        # Load binary tile
        with rasterio.open(bin_path) as src:
            binary  = src.read(1).astype(np.uint8)
            profile = src.profile.copy()

        lupine_before = int((binary == 1).sum())

        # Apply postprocessing
        binary = postprocess_tile(
            binary, profile, cultivated_gdf,
            shadow_path=shd_path if (APPLY_SHADOW_MASK and os.path.exists(shd_path)) else None,
            median_size=MEDIAN_SIZE,
            opening_size=OPENING_SIZE,
            closing_size=CLOSING_SIZE,
         )

        lupine_after = int((binary == 1).sum())
        removed     += lupine_before - lupine_after

        # Skip if no lupine remains after cleaning
        if lupine_after == 0:
            del binary; gc.collect()
            continue

        # Save postprocessed tile
        pp_profile = profile.copy()
        pp_profile.update(count=1, dtype="uint8", compress="lzw",
                          tiled=True, blockxsize=256, blockysize=256,
                          nodata=None)
        with rasterio.open(pp_path, "w", **pp_profile) as dst:
            dst.write(binary[np.newaxis, :, :])
        del binary; gc.collect()

    # Progress
    elapsed = time.time() - t0
    done    = i + 1
    rate    = done / elapsed if elapsed > 0 else 0
    eta     = remaining_count / rate if rate > 0 else 0

    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == len(to_process):
        print(f"\n  ── Checkpoint [{i+1}/{len(to_process)}] ──────────────────")
        print(f"     Elapsed: {elapsed/3600:.1f}h  |  ETA: {eta/3600:.1f}h")
        for thresh in PIXEL_THRESHOLDS:
            n = len(os.listdir(pp_tile_dir(thresh)))
            print(f"     PP tiles t={thresh}: {n}")
        print()

print(f"\n  Done. Time: {(time.time()-t0)/3600:.1f}h")
print(f"  Lupine pixels removed by postprocessing: {removed:,}")
for thresh in PIXEL_THRESHOLDS:
    n = len(os.listdir(pp_tile_dir(thresh)))
    print(f"  PP tiles t={thresh}: {n} → {pp_tile_dir(thresh)}")

In [ ]:
# ── Coverage summary ──────────────────────────────────────────────────────────
print("── Coverage summary ─────────────────────────────────────────")

pixel_area_m2 = 0.25 * 0.25  # 0.25m resolution

for thresh in PIXEL_THRESHOLDS:
    pp_dir  = pp_tile_dir(thresh)
    pp_tiles = [f for f in os.listdir(pp_dir) if f.endswith(".tif")]
    total_lupine = 0
    for fname in pp_tiles:
        with rasterio.open(os.path.join(pp_dir, fname)) as src:
            data = src.read(1)
        total_lupine += int((data == 1).sum())
    area_ha = total_lupine * pixel_area_m2 / 10000
    print(f"  t={thresh}: {len(pp_tiles)} tiles  |  {total_lupine:,} lupine px  |  {area_ha:.2f} ha")